[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/stabilizer/stabilizer.ipynb)

# Two Qubits Kept in Four

Four qubits store two, so that any single qubit going wrong is noticed, whether it is flipped, phase-flipped, or both. Two checks do it: whether flipping all four qubits leaves the state alone, and whether phase-flipping all four does. A stored state passes both; any single error fails at least one, and the run is discarded. Errors are caught rather than corrected, so a kept run can only be wrong when two errors happened together.

The whole of Cl(4, 4) is at work here. A state of four qubits is sixteen real numbers, a spinor, and the operators on it are the algebra itself: its two hundred and fifty-six dimensions are exactly the linear maps of sixteen numbers to sixteen numbers. Every pattern of bit and phase flips on the four qubits is, up to sign, a single blade, and the two hundred and fifty-six patterns are all two hundred and fifty-six blades. The checks, the errors and the flips of the two stored qubits are blades, and the projector onto the code is a sum of four of them.

In stabilizer notation this is the $[[4, 2, 2]]$ code: its stabilizer group is generated by $XXXX$ and $ZZZZ$, and its logical operators are $\bar X_1 = XXII$, $\bar Z_1 = ZIZI$, $\bar X_2 = XIXI$ and $\bar Z_2 = ZZII$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.animation import save_figure
from examples.quantum.stabilizer import render

ga = Algebra("a+b+c+d+e-f-g-h-")
mv = NumpyContext(ga).multivector
exact = ga.exact.multivector

Scalar = ga.gatype.scalar()
Full = ga.gatype.full()
State = ga.gatype.from_blades("1 b c d h bc bd cd bh ch dh bcd bch bdh cdh bcdh")

# A state of four qubits is sixteen real numbers: a spinor, held in the algebra as an element times an
# idempotent of four commuting involutions. An operator multiplies it on the left, and the readout takes
# its sixteen components back out.
ideal = (1 + exact.a) * (1 + exact.be) * (1 + exact.cf) * (1 + exact.dg) / 16
embedding = State * ideal                                                      # [] Full <- State
readout = (16 * Full).cast(State)                                              # [] State <- Full
action = readout(Full * embedding)                                             # [] State <- (Full, State)
# The positive definite pairing of states, preserved by the action.
pairing = 16 * exact.cdeh.scalar_product(embedding.reverse() * exact.efgh * embedding)  # [] Scalar <- (State, State)

# Each qubit's bit flip and phase flip: blades that anticommute on their own qubit and commute across qubits.
flips = stack([mv.b, -mv.bce, -mv.bcdef, -mv.ah])                              # [qubits] Full
phase_flips = stack([mv.be, mv.cf, mv.dg, -mv.abcdefg])                        # [qubits] Full
# What can happen to each qubit: nothing, a bit flip, a phase flip, or both.
errors = stack([mv.scalar().broadcast_to(flips.shape), flips, phase_flips, flips * phase_flips], axis=-1)  # [qubits, kinds] Full
# All four qubits 0.
ground = mv.scalar().cast(State)                                               # [] State

# The angles of the two stored qubits, and the chances of an error on each qubit.
angles = np.array([0.7, 1.9])
rates = np.linspace(0.0, 0.5, 51)

## 1. Two checks

Each check is a product of four blades, one per qubit. A qubit's flip and phase flip anticommute, and the two checks meet on all four qubits, an even number, so the checks commute. The states both checks leave alone form the code, picked out by multiplying the projectors `(1 + check) / 2`; the four sign patterns of the two checks give four projectors that together are the identity. The flips and phase flips of the two stored qubits act on pairs of qubits, so they too meet each check on an even number of qubits and commute with both.

In [ ]:
# The two checks: flipping all four qubits, and phase-flipping all four.
checks = stack([reduce(mul, flips), reduce(mul, phase_flips)])                 # [checks] Full
# Each outcome of the two checks, and the projector onto the states that give it; the first is the code.
outcomes = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]])                      # [outcomes, checks]
halves = (1 + checks * outcomes) / 2                                           # [outcomes, checks] Full
projectors = halves[:, 0] * halves[:, 1]                                       # [outcomes] Full
code = projectors[0]                                                           # [] Full

# The flips and phase flips of the two stored qubits: pairs of qubits, commuting with both checks.
logical_flips = stack([flips[0] * flips[1], flips[0] * flips[2]])              # [logical] Full
logical_phase_flips = stack([phase_flips[0] * phase_flips[2], phase_flips[0] * phase_flips[1]])  # [logical] Full

## 2. Two qubits stored

A single qubit turns from 0 towards 1 by a rotor in the plane of its flip and phase flip. A stored qubit turns the same way, by a rotor in the plane of its logical flip and phase flip. Projecting all-zeros onto the code and turning it by both rotors stores the two qubits at their angles. Both checks read $+1$ on the result, and each stored qubit's flip and phase flip read back the sine and cosine of its angle.

In [ ]:
def expectation(operators: Full, states: State) -> Scalar:
    """The expectation of each operator on each state."""
    return pairing(states, action(operators, states)) / pairing(states, states)  # [...] Scalar


def turned(start: State, qubit_flips: Full, qubit_phase_flips: Full, angles: np.ndarray) -> State:
    """The start state with each of two qubits turned by its angle from 0 towards 1, by a rotor in the
    plane of its flip and phase flip."""
    turns = ((qubit_flips * qubit_phase_flips) * (angles / 2)).exp()           # [..., 2] Full
    return action(turns[..., 0] * turns[..., 1], start)                        # [...] State


start = action(code, ground)                                                   # [] State
start = start / pairing(start, start).square_root()
stored = turned(start, logical_flips, logical_phase_flips, angles)             # [] State
logical_operators = stack([logical_flips, logical_phase_flips], axis=-1)       # [logical, 2] Full

check_readings = expectation(checks, stored[..., None])                        # [checks] Scalar
logical_readings = expectation(logical_operators, stored[..., None, None])     # [logical, 2] Scalar

print("checks:", render.scalars(check_readings))
print("flip and phase flip of each stored qubit:\n", render.scalars(logical_readings))
print("sine and cosine of each angle:\n", np.stack([np.sin(angles), np.cos(angles)], axis=-1))

## 3. Every single error is caught

A bit flip on any qubit anticommutes with the phase-flip check, a phase flip with the bit-flip check, and both at once with both checks. Each changes the reading of at least one check, so every column of the table but the first has a $-1$ in it. Two errors can leave both checks alone, as two bit flips do; that is why the code catches single errors rather than correcting them.

In [ ]:
readings = expectation(checks, action(errors, stored)[..., None])              # [qubits, kinds, checks] Scalar
render.print_readings(readings)

## 4. Under noise

Each qubit is left alone, or hit by one of the three errors with equal chance, independently of the others. The patterns of errors on the four qubits are the blades of the algebra, all two hundred and fifty-six of them, and they form one batch, each weighted by its probability. The chance that both checks pass is the part of the corrupted state left in the code, and the runs that fail are discarded. A kept run is wrong only when errors on two or more qubits went unnoticed together, so how often it is wrong grows with the square of the error rate, where two bare qubits under the same noise go wrong in proportion to it. The price is the discarded runs, the dashed line.

In matrix notation Cl(4, 4) is the algebra of real $16 \times 16$ matrices, and each blade is a tensor product of four of the real matrices $I$, $X$, $Z$ and $XZ$, one per qubit.

In [ ]:
def noise(qubits: int, rates: np.ndarray) -> tuple[Full, np.ndarray]:
    """Every pattern of errors on the first so many qubits, and its probability at each rate: each qubit
    is left alone, or hit by one of the three errors with equal chance."""
    kinds = np.indices((errors.shape[-1],) * qubits)                           # [qubits, kinds...]
    chances = np.stack([1 - rates, rates / 3, rates / 3, rates / 3], axis=-1)  # [rates, kinds]
    patterns = reduce(mul, (errors[qubit, kind] for qubit, kind in enumerate(kinds)))  # [kinds...] Full
    weights = reduce(mul, (chances[:, kind] for kind in kinds))                # [rates, kinds...]
    return patterns, weights


def averaged(values: Scalar, weights: np.ndarray) -> Scalar:
    """The average over error patterns, the trailing axes of the values, at each rate."""
    return (values * weights).sum(axis=tuple(range(1 - weights.ndim, 0)))      # [rates] Scalar


patterns, weights = noise(errors.shape[0], rates)
corrupted = action(patterns, stored)                                           # [kinds, kinds, kinds, kinds] State
# The chance that both checks pass is the part of the state left in the code.
passed = pairing(corrupted, action(code, corrupted))                           # [kinds, kinds, kinds, kinds] Scalar
unchanged = pairing(stored, corrupted).squared()                               # [kinds, kinds, kinds, kinds] Scalar
kept = averaged(passed, weights)                                               # [rates] Scalar
encoded_errors = 1 - averaged(unchanged, weights) / kept                       # [rates] Scalar

# Two bare qubits at the same angles, under the same noise.
bare = turned(ground, flips[:2], phase_flips[:2], angles)                      # [] State
bare_patterns, bare_weights = noise(2, rates)
bare_errors = 1 - averaged(pairing(bare, action(bare_patterns, bare)).squared(), bare_weights)  # [rates] Scalar

save_figure(render.draw_noise(rates, kept, encoded_errors, bare_errors), "stabilizer_noise");

In [ ]:
# checks
assert np.allclose(render.scalars(check_readings), 1)
assert np.allclose(render.scalars(logical_readings), np.stack([np.sin(angles), np.cos(angles)], axis=-1))
assert (render.scalars(readings)[:, 1:].min(axis=-1) < 0).all()
# Doubling a small rate quadruples how often kept runs go wrong, and doubles it for bare qubits.
assert np.isclose(render.scalars(encoded_errors)[2] / render.scalars(encoded_errors)[1], 4, rtol=0.05)
assert np.isclose(render.scalars(bare_errors)[2] / render.scalars(bare_errors)[1], 2, rtol=0.05)